# 06: Deterministic PDE Solver

Demonstrates deterministic advection PDE solving with automatic numerical scheme selection.

In [ ]:
# Setup and imports
import sys, os
repo_root = os.path.dirname(os.getcwd())
src_path = os.path.join(repo_root, "src")
if os.path.isdir(src_path) and src_path not in sys.path:
    sys.path.insert(0, src_path)

import numpy as np
import matplotlib.pyplot as plt
from stochlib import configure_logging, get_logger
import logging

# Configure logging
configure_logging(level=logging.INFO, verbose=True)
logger = get_logger("notebook")
logger.info("Starting deterministic solver notebook")

In [ ]:
# Import deterministic solver components
from stochlib.setup import Grid, InitialCondition
from stochlib.deterministic import DeterministicPDESolver, solve_deterministic_pde

logger.info("Imported deterministic solver components")

## Part 1: Automatic Scheme Selection

In [ ]:
# Setup grid and initial condition
grid = Grid(x_start=-5.0, x_end=5.0, num_points_x=128)
ic = InitialCondition(grid=grid, func_type="gaussian", x0=0.0, sigma_x=0.5)

# Time discretization
t_eval = np.linspace(0, 2.0, 101)

# Define drift velocity (constant rightward advection)
def drift(x):
    """Constant drift velocity"""
    return 1.0 * np.ones_like(x)

logger.info(f"Grid: {grid.num_points_x} points in [{grid.x_start}, {grid.x_end}]")
logger.info(f"Time: {len(t_eval)-1} steps from t=0 to t={t_eval[-1]}")

In [ ]:
# Solve with automatic scheme selection
logger.info("Solving with automatic scheme selection...")
t, u = solve_deterministic_pde(
    grid=grid,
    ic=ic,
    t_eval=t_eval,
    drift=drift,
    scheme="auto",  # Automatically choose based on CFL number
    boundary_mode="none",
    verbose=False,
)

logger.info(f"Solve complete. Solution shape: {u.shape}")
print(f"Initial distribution peak: {np.max(ic.f0):.6f}")
print(f"Final distribution peak: {np.max(u[-1]):.6f}")
print(f"Mass conservation: {np.sum(u[-1]) * grid.dx:.6f}")

In [ ]:
# Plot solution evolution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: Solution at different times
time_indices = [0, 25, 50, 75, 100]
colors = plt.cm.viridis(np.linspace(0, 1, len(time_indices)))

for idx, time_idx in enumerate(time_indices):
    axes[0].plot(grid.x_grid, u[time_idx], color=colors[idx], 
                 label=f't={t[time_idx]:.2f}', linewidth=2, alpha=0.8)
axes[0].set_xlabel('x', fontsize=12)
axes[0].set_ylabel('Solution u(x,t)', fontsize=12)
axes[0].set_title('Advection PDE: Solution Evolution', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Right: Final solution with initial
axes[1].plot(grid.x_grid, ic.f0, 'b-', linewidth=2, label='Initial', alpha=0.8)
axes[1].plot(grid.x_grid, u[-1], 'r-', linewidth=2, label='Final', alpha=0.8)
axes[1].set_xlabel('x', fontsize=12)
axes[1].set_ylabel('Solution u(x,t)', fontsize=12)
axes[1].set_title(f'Initial vs Final (t={t[-1]:.1f})', fontsize=13)
axes[1].legend(fontsize=11)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

logger.info("Plotted solution evolution")

## Part 2: Scheme Comparison

In [ ]:
# Compare upwind and Lax-Wendroff schemes
schemes = ["upwind", "lax_wendroff"]
results = {}

for scheme in schemes:
    logger.info(f"Solving with {scheme} scheme...")
    t_scheme, u_scheme = solve_deterministic_pde(
        grid=grid,
        ic=ic,
        t_eval=t_eval,
        drift=drift,
        scheme=scheme,
        boundary_mode="none",
        verbose=False,
    )
    results[scheme] = u_scheme

logger.info("Scheme comparison complete")

In [ ]:
# Plot scheme comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, scheme in enumerate(schemes):
    # Plot multiple time snapshots
    time_indices = [0, 25, 50, 75, 100]
    colors = plt.cm.viridis(np.linspace(0, 1, len(time_indices)))
    
    for i, time_idx in enumerate(time_indices):
        axes[idx].plot(grid.x_grid, results[scheme][time_idx], 
                      color=colors[i], label=f't={t[time_idx]:.1f}', 
                      linewidth=2, alpha=0.8)
    
    axes[idx].set_xlabel('x', fontsize=11)
    axes[idx].set_ylabel('u(x,t)', fontsize=11)
    scheme_name = scheme.replace('_', '-').title()
    axes[idx].set_title(f'{scheme_name} Scheme', fontsize=12)
    axes[idx].legend(fontsize=9, loc='upper right')
    axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

logger.info("Plotted scheme comparison")

## Part 3: Step-by-Step Manual Solving

In [ ]:
# Create solver object for manual control
solver = DeterministicPDESolver(
    grid=grid,
    ic=ic,
    drift=drift,
    scheme="auto",
    boundary_mode="none",
)

logger.info("Created solver for step-by-step integration")

# Manual stepping
dt = 0.02
n_steps = 50

snapshots = []
times = []

for i in range(n_steps):
    if i % 10 == 0:
        snapshots.append(solver.u.copy())
        times.append(solver.t_current)
    solver.step(dt)

# Add final snapshot
snapshots.append(solver.u.copy())
times.append(solver.t_current)

logger.info(f"Manual stepping complete: {len(snapshots)} snapshots")

In [ ]:
# Plot manual stepping results
fig, ax = plt.subplots(figsize=(10, 5))

cmap = plt.colormaps['viridis']
for idx, (t_snap, u_snap) in enumerate(zip(times, snapshots)):
    frac = idx / (len(times) - 1) if len(times) > 1 else 0
    ax.plot(grid.x_grid, u_snap, color=cmap(frac), 
            label=f't={t_snap:.2f}', linewidth=2, alpha=0.7)

ax.set_xlabel('x', fontsize=12)
ax.set_ylabel('u(x,t)', fontsize=12)
ax.set_title('Manual Step-by-Step Solving (dt=0.02)', fontsize=13)
ax.legend(fontsize=10, loc='best')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

logger.info("Plotted manual stepping results")

## Part 4: CFL Stability Analysis

In [ ]:
# Demonstrate CFL constraint with large time step
logger.info("Testing CFL stability with large dt...")

# Large time step violates CFL condition
t_coarse = np.linspace(0, 2.0, 11)  # Only 10 steps

t_large, u_large = solve_deterministic_pde(
    grid=grid,
    ic=ic,
    t_eval=t_coarse,
    drift=drift,
    scheme="auto",
    boundary_mode="none",
    verbose=False,
)

logger.info("Large time step solve complete")
print(f"\nCFL Analysis:")
print(f"  Grid spacing (dx): {grid.dx:.4f}")
print(f"  Drift velocity (c): {1.0:.4f}")
print(f"  Time step (dt): {t_coarse[1]-t_coarse[0]:.4f}")
print(f"  CFL number (c*dt/dx): {1.0 * (t_coarse[1]-t_coarse[0]) / grid.dx:.4f}")
print(f"  Stable if CFL ≤ 1.0: {'✓ Stable' if 1.0 * (t_coarse[1]-t_coarse[0]) / grid.dx <= 1.0 else '✗ Unstable'}")

In [ ]:
# Compare fine vs coarse time stepping
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Fine time stepping
axes[0].plot(grid.x_grid, ic.f0, 'b--', linewidth=2, label='Initial', alpha=0.7)
axes[0].plot(grid.x_grid, u[-1], 'r-', linewidth=2, label='Final (100 steps)', alpha=0.8)
axes[0].set_xlabel('x', fontsize=11)
axes[0].set_ylabel('u(x,t)', fontsize=11)
axes[0].set_title('Fine Time Stepping (dt=0.02, CFL=0.25)', fontsize=12)
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# Coarse time stepping
axes[1].plot(grid.x_grid, ic.f0, 'b--', linewidth=2, label='Initial', alpha=0.7)
axes[1].plot(grid.x_grid, u_large[-1], 'r-', linewidth=2, label='Final (10 steps)', alpha=0.8)
axes[1].set_xlabel('x', fontsize=11)
axes[1].set_ylabel('u(x,t)', fontsize=11)
axes[1].set_title('Coarse Time Stepping (dt=0.2, CFL=2.54)', fontsize=12)
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

logger.info("Plotted fine vs coarse comparison")

## Summary

This notebook demonstrated:
1. **Automatic Scheme Selection**: Solver automatically chooses between upwind and Lax-Wendroff based on CFL number
2. **Scheme Comparison**: Different numerical schemes handle advection differently
3. **Manual Step-by-Step Solving**: Direct control over integration via solver object
4. **CFL Stability**: Understanding the relationship between grid spacing, time step, and advection velocity

Key Concepts:
- **CFL Condition**: For explicit schemes, `c·dt/dx ≤ 1` ensures stability
- **Upwind Scheme**: More diffusive, always stable
- **Lax-Wendroff**: Less diffusive, can oscillate if CFL > 1